In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
df = pd.read_hdf("pems-bay.h5")

traffic = df.values

print(traffic.shape)

(52116, 325)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(52104, 12, 325)
(52104, 325)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn


class TransformerModel(nn.Module):

    def __init__(
        self,
        num_nodes=325,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            num_nodes,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(
            d_model,
            num_nodes
        )

    def forward(self,x):

        x = self.input_proj(x)

        x = self.transformer(x)

        x = x[:,-1,:]

        x = self.fc(x)

        return x

In [9]:
model = TransformerModel()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 325])
torch.Size([64, 325])


In [10]:
model = TransformerModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.013756
Epoch 2/30 Loss: 0.003916
Epoch 3/30 Loss: 0.003215
Epoch 4/30 Loss: 0.002726
Epoch 5/30 Loss: 0.002358
Epoch 6/30 Loss: 0.002069
Epoch 7/30 Loss: 0.001866
Epoch 8/30 Loss: 0.001684
Epoch 9/30 Loss: 0.001550
Epoch 10/30 Loss: 0.001442
Epoch 11/30 Loss: 0.001350
Epoch 12/30 Loss: 0.001289
Epoch 13/30 Loss: 0.001233
Epoch 14/30 Loss: 0.001218
Epoch 15/30 Loss: 0.001162
Epoch 16/30 Loss: 0.001125
Epoch 17/30 Loss: 0.001108
Epoch 18/30 Loss: 0.001079
Epoch 19/30 Loss: 0.001073
Epoch 20/30 Loss: 0.001058
Epoch 21/30 Loss: 0.001054
Epoch 22/30 Loss: 0.001021
Epoch 23/30 Loss: 0.001016
Epoch 24/30 Loss: 0.001010
Epoch 25/30 Loss: 0.000996
Epoch 26/30 Loss: 0.000986
Epoch 27/30 Loss: 0.000983
Epoch 28/30 Loss: 0.000971
Epoch 29/30 Loss: 0.000972
Epoch 30/30 Loss: 0.000961


In [11]:
test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.024251356720924377
RMSE: 0.038978272767449376
